## Repair ORCID-Fusion & Displacement Waves — oxjob #608 Phase 1 wind-down

Two authority-evidence repair waves over `openalex.works.work_authors`, run manually in the
end2end quiet window (same discipline as RepairMisboundAuthorships):

- **waveO_orcid_split** — profiles fusing >=3 distinct qualified ORCIDs (support >=2 seats each):
  minority-ORCID seats rebind to the unique other profile supporting that ORCID, else NULL
  (queue-don't-mint). ORCID-less seats untouched.
- **waveD_displacement** — profiles whose incompatible seats are >=80% explained by off-by-one
  displacement (own name at seat +/-1): realign each seat to the unique on-work donor profile
  whose name matches the seat's raw name and which is itself misplaced; else NULL.

Flow: candidates (live rebuild) -> batch admission (curation-excluded, log-deduped,
profile-throttled) -> 150/tier opus anchor gate -> gated apply (optimistic `<=>` MERGE) ->
pending_sync -> verify. Gate must read >=98% per applied tier before an apply run.

In [ ]:
-- Job mode: knobs arrive as job parameters (Repair Orcid Displacement job).
-- Unknown wave -> empty batch; repair_apply anything but 'true' -> dry run.
DECLARE OR REPLACE VARIABLE repair_wave STRING DEFAULT :repair_wave;

In [ ]:
-- Master apply gate. FALSE = resolve/gate only (read-only).
DECLARE OR REPLACE VARIABLE repair_apply BOOLEAN DEFAULT (:repair_apply = 'true');

In [ ]:
-- Throttle: max profiles admitted per batch.
DECLARE OR REPLACE VARIABLE batch_max_profiles BIGINT DEFAULT CAST(:batch_max_profiles AS BIGINT);

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.oxjob608_authority_repair_log (
    applied_at TIMESTAMP,
    wave STRING,
    work_id BIGINT,
    author_sequence INT,
    raw_author_name STRING,
    old_author_id BIGINT,
    new_author_id BIGINT,
    tier STRING
) USING DELTA

In [ ]:
-- Wave O candidates: fused-profile minority-ORCID seats, live rebuild each run.
CREATE OR REPLACE TABLE openalex.authors.oxjob608_waveO_candidates AS
WITH seats AS (
    SELECT w.id AS work_id, t.pos AS author_sequence,
           NULLIF(TRIM(t.a.raw_orcid), '') AS raw_orcid,
           TRIM(t.a.raw_author_name) AS raw_name
    FROM openalex.works.openalex_works_base w
    LATERAL VIEW posexplode(w.authorships) t AS pos, a
    WHERE EXISTS(w.authorships, x -> NULLIF(TRIM(x.raw_orcid), '') IS NOT NULL)
),
quality AS (
    -- disqualify shared/junk ORCIDs: > 3 surname families
    SELECT s.raw_orcid
    FROM seats s
    LEFT JOIN openalex.authors.author_names an ON s.raw_name = an.raw_author_name
    WHERE s.raw_orcid IS NOT NULL
    GROUP BY s.raw_orcid
    HAVING COUNT(DISTINCT an.match_last) <= 3
),
bound AS (
    SELECT s.work_id, s.author_sequence, s.raw_orcid, s.raw_name, wa.author_id
    FROM seats s
    JOIN quality q ON s.raw_orcid = q.raw_orcid
    JOIN openalex.works.work_authors wa
      ON s.work_id = wa.work_id AND s.author_sequence = wa.author_sequence
    WHERE wa.author_id IS NOT NULL
),
support AS (
    SELECT author_id, raw_orcid, COUNT(*) AS n
    FROM bound GROUP BY author_id, raw_orcid HAVING COUNT(*) >= 2
),
fused AS (
    SELECT author_id FROM support GROUP BY author_id HAVING COUNT(*) >= 3
),
majority AS (
    SELECT author_id, raw_orcid AS majority_orcid
    FROM (SELECT author_id, raw_orcid,
                 ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY n DESC, raw_orcid) AS rk
          FROM support)
    WHERE rk = 1
),
owners AS (
    -- for each (orcid, fused-profile): the unique OTHER profile supporting that orcid.
    -- Self-exclusion matters: the fused profile itself supports its minority orcids.
    SELECT b.raw_orcid, b.author_id AS from_profile,
           CASE WHEN COUNT(DISTINCT s2.author_id) = 1 THEN MAX(s2.author_id) END AS sole_other_owner,
           MAX(s2.n) AS owner_support
    FROM (SELECT DISTINCT author_id, raw_orcid FROM support) b
    JOIN support s2 ON s2.raw_orcid = b.raw_orcid AND s2.author_id != b.author_id
    GROUP BY b.raw_orcid, b.author_id
)
-- Wave O acts ONLY on rebindable seats: minority-ORCID seats whose ORCID has exactly one
-- other supported profile. No NULLs in this wave — unbinding name-matching seats on ORCID
-- evidence alone is deferred (duplicate-ORCID registrations / depositor errors confound it).
SELECT b.work_id, b.author_sequence, b.raw_name, b.author_id AS live_author_id,
       b.raw_orcid,
       o.sole_other_owner AS rebind_author_id,
       'orcid_rebind' AS tier,
       b.author_id AS batch_profile_id,
       sp.n AS seat_orcid_support,
       o.owner_support
FROM bound b
JOIN fused f ON b.author_id = f.author_id
JOIN majority m ON b.author_id = m.author_id
JOIN support sp ON sp.author_id = b.author_id AND sp.raw_orcid = b.raw_orcid
JOIN owners o ON o.raw_orcid = b.raw_orcid AND o.from_profile = b.author_id
-- ORCID and NAME must agree: the receiving profile's name must be compatible with the
-- seat's name. Disagreement = likely depositor ORCID paste-error — dropped, not applied.
LEFT JOIN openalex.authors.openalex_authors po ON o.sole_other_owner = po.id
LEFT JOIN openalex.authors.authors pl ON o.sole_other_owner = pl.id
LEFT JOIN openalex.authors.author_names an_s ON b.raw_name = an_s.raw_author_name
LEFT JOIN openalex.authors.author_names an_o
    ON TRIM(COALESCE(po.display_name, pl.display_name)) = an_o.raw_author_name
WHERE b.raw_orcid != m.majority_orcid
  AND o.sole_other_owner IS NOT NULL
  AND COALESCE(openalex.authors.names_compatible(
      an_s.match_last, an_s.match_first, an_o.match_last, an_o.match_first,
      b.raw_name, COALESCE(po.display_name, pl.display_name)), false)

In [ ]:
-- Wave D candidates: off-by-one displaced profiles, donor realign on-work.
CREATE OR REPLACE TABLE openalex.authors.oxjob608_waveD_candidates AS
WITH pkeys AS (
    -- profile identity anchor = display_name (full_name fallback) — detector semantics
    SELECT p.id, COALESCE(NULLIF(TRIM(p.display_name), ''), TRIM(p.full_name)) AS fn,
           an.match_last AS p_last, an.match_first AS p_first
    FROM openalex.authors.openalex_authors p
    JOIN openalex.authors.author_names an
      ON COALESCE(NULLIF(TRIM(p.display_name), ''), TRIM(p.full_name)) = an.raw_author_name
    WHERE COALESCE(NULLIF(TRIM(p.display_name), ''), TRIM(p.full_name)) IS NOT NULL
      AND an.match_last IS NOT NULL
),
bad AS (
    SELECT wa.work_id, wa.author_sequence, TRIM(wa.raw_author_name) AS raw_name,
           wa.author_id, pk.p_last, pk.p_first, pk.fn,
           an_r.match_last AS r_last, an_r.match_first AS r_first
    FROM openalex.works.work_authors wa
    JOIN pkeys pk ON wa.author_id = pk.id
    JOIN openalex.authors.author_names an_r ON TRIM(wa.raw_author_name) = an_r.raw_author_name
    WHERE wa.author_id IS NOT NULL AND an_r.match_last IS NOT NULL
      -- CJK abstention: frozen-parser gap, same exclusion as the misbinding detector
      AND NOT (wa.raw_author_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]'
               OR pk.fn RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]')
      AND NOT COALESCE(openalex.authors.names_compatible(
          an_r.match_last, an_r.match_first, pk.p_last, pk.p_first,
          wa.raw_author_name, pk.fn), false)
),
explained AS (
    SELECT b.author_id, b.work_id, b.author_sequence,
        MAX(CASE WHEN an_n.match_last = b.p_last
                  AND (an_n.match_first IS NULL OR b.p_first IS NULL
                       OR LEFT(an_n.match_first, 1) = LEFT(b.p_first, 1))
             THEN 1 ELSE 0 END) AS off_by_one
    FROM bad b
    JOIN openalex.works.work_authors nb
      ON nb.work_id = b.work_id
     AND nb.author_sequence BETWEEN b.author_sequence - 1 AND b.author_sequence + 1
     AND nb.author_sequence != b.author_sequence
    JOIN openalex.authors.author_names an_n ON TRIM(nb.raw_author_name) = an_n.raw_author_name
    GROUP BY b.author_id, b.work_id, b.author_sequence
),
admitted_profiles AS (
    SELECT author_id
    FROM explained
    GROUP BY author_id
    HAVING COUNT(*) >= 3 AND SUM(off_by_one) >= 0.8 * COUNT(*)
),
-- donors: misplaced profiles bound on the same work whose NAME matches the bad seat's name
donor_cand AS (
    SELECT b.work_id, b.author_sequence, b.raw_name, b.author_id AS live_author_id,
           d.author_id AS donor_id
    FROM bad b
    JOIN admitted_profiles ap ON b.author_id = ap.author_id
    JOIN bad d ON d.work_id = b.work_id             -- donor must itself be misplaced (chain member)
    JOIN pkeys dpk ON d.author_id = dpk.id
    WHERE d.author_id != b.author_id
      -- STRICT donor match: surname equality + given equality, or exact initial
      -- consistency when one side is an initial (no prefix logic — romanized-CJK
      -- given names defeat it; #608 counter-rule)
      AND dpk.p_last = b.r_last
      AND (dpk.p_first = b.r_first
           OR (LENGTH(dpk.p_first) = 1 AND LENGTH(b.r_first) > 1 AND dpk.p_first = LEFT(b.r_first, 1))
           OR (LENGTH(b.r_first) = 1 AND LENGTH(dpk.p_first) > 1 AND b.r_first = LEFT(dpk.p_first, 1)))
      AND dpk.p_first IS NOT NULL AND b.r_first IS NOT NULL
),
unique_donors AS (
    SELECT work_id, author_sequence, raw_name, live_author_id, MAX(donor_id) AS donor_id
    FROM donor_cand
    GROUP BY work_id, author_sequence, raw_name, live_author_id
    HAVING COUNT(DISTINCT donor_id) = 1
),
exclusive AS (
    -- each donor may claim exactly one seat per work
    SELECT *,
        COUNT(*) OVER (PARTITION BY work_id, donor_id) AS donor_claims
    FROM unique_donors
)
SELECT b.work_id, b.author_sequence, b.raw_name, b.author_id AS live_author_id,
       CAST(NULL AS STRING) AS raw_orcid,
       e.donor_id AS rebind_author_id,
       CASE WHEN e.donor_id IS NOT NULL THEN 'displace_realign' ELSE 'displace_null' END AS tier,
       b.author_id AS batch_profile_id,
       CAST(NULL AS BIGINT) AS seat_orcid_support,
       CAST(NULL AS BIGINT) AS owner_support
FROM bad b
JOIN admitted_profiles ap ON b.author_id = ap.author_id
LEFT JOIN exclusive e
  ON b.work_id = e.work_id AND b.author_sequence = e.author_sequence AND e.donor_claims = 1

In [ ]:
-- Wave I candidates: hyper-work generalized repair — ANY incompatible seat on a
-- >=100-author work (no profile-level displacement bar; the Taylor-specimen class).
-- Precision replaces the 80% bar with: judged same-person exclusion (gemini + rules
-- verdict overlays kill reorder/citation-variant FPs), CJK abstention, hyper scope
-- (excludes name-alike overmerge on small works), and per-tier opus gate.
-- Tiers: onwork_realign = unique strict-match donor that is ITSELF misplaced on the
-- same work (never moves a correctly-seated profile); hyper_null = donor-less ->
-- unbind, matcher re-attaches (wave-N semantics, hyper-scoped).
CREATE OR REPLACE TABLE openalex.authors.oxjob608_waveI_candidates AS
WITH hyper AS (
    SELECT work_id FROM openalex.works.work_authors
    GROUP BY work_id HAVING COUNT(*) >= 100
),
pkeys AS (
    SELECT p.id, COALESCE(NULLIF(TRIM(p.display_name), ''), TRIM(p.full_name)) AS fn,
           an.match_last AS p_last, an.match_first AS p_first
    FROM openalex.authors.openalex_authors p
    JOIN openalex.authors.author_names an
      ON COALESCE(NULLIF(TRIM(p.display_name), ''), TRIM(p.full_name)) = an.raw_author_name
    WHERE COALESCE(NULLIF(TRIM(p.display_name), ''), TRIM(p.full_name)) IS NOT NULL
      AND an.match_last IS NOT NULL
),
bad AS (
    SELECT wa.work_id, wa.author_sequence, TRIM(wa.raw_author_name) AS raw_name,
           wa.author_id, pk.p_last, pk.p_first, pk.fn,
           an_r.match_last AS r_last, an_r.match_first AS r_first
    FROM openalex.works.work_authors wa
    JOIN hyper h ON wa.work_id = h.work_id
    JOIN pkeys pk ON wa.author_id = pk.id
    JOIN openalex.authors.author_names an_r ON TRIM(wa.raw_author_name) = an_r.raw_author_name
    WHERE wa.author_id IS NOT NULL AND an_r.match_last IS NOT NULL
      -- CJK abstention: frozen-parser gap, same exclusion as the misbinding detector
      AND NOT (wa.raw_author_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]'
               OR pk.fn RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]')
      AND NOT COALESCE(openalex.authors.names_compatible(
          an_r.match_last, an_r.match_first, pk.p_last, pk.p_first,
          wa.raw_author_name, pk.fn), false)
      -- judged same-person pairs are measurement FPs, never misbindings
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_gemini j
                      WHERE j.full_name = pk.fn AND j.raw_name = TRIM(wa.raw_author_name)
                        AND j.judge_json:same_person = 'true')
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_rules jr
                      WHERE jr.full_name = pk.fn AND jr.raw_name = TRIM(wa.raw_author_name)
                        AND jr.same_person)
),
donor_cand AS (
    SELECT b.work_id, b.author_sequence, b.raw_name, b.author_id AS live_author_id,
           d.author_id AS donor_id
    FROM bad b
    JOIN bad d ON d.work_id = b.work_id            -- donor must itself be misplaced
    JOIN pkeys dpk ON d.author_id = dpk.id
    WHERE d.author_id != b.author_id
      -- STRICT donor match: surname equality + given equality, or exact initial
      -- consistency when one side is an initial (no prefix logic; #608 counter-rule)
      AND dpk.p_last = b.r_last
      AND (dpk.p_first = b.r_first
           OR (LENGTH(dpk.p_first) = 1 AND LENGTH(b.r_first) > 1 AND dpk.p_first = LEFT(b.r_first, 1))
           OR (LENGTH(b.r_first) = 1 AND LENGTH(dpk.p_first) > 1 AND b.r_first = LEFT(dpk.p_first, 1)))
      AND dpk.p_first IS NOT NULL AND b.r_first IS NOT NULL
      -- multi-initial consistency: keys truncate to the first initial (Class 7),
      -- so 'Liu, X. Y.' would match donor 'X. K. Liu' — when either side carries
      -- >=2 initials, require the full initial sequences to agree
      AND (
        ARRAY_JOIN(FILTER(SPLIT(REGEXP_REPLACE(LOWER(b.raw_name), '[.,]', ' '), ' +'), t -> LENGTH(t) = 1), '') = ''
        OR ARRAY_JOIN(FILTER(SPLIT(REGEXP_REPLACE(LOWER(dpk.fn), '[.,]', ' '), ' +'), t -> LENGTH(t) = 1), '') = ''
        OR ARRAY_JOIN(FILTER(SPLIT(REGEXP_REPLACE(LOWER(b.raw_name), '[.,]', ' '), ' +'), t -> LENGTH(t) = 1), '')
           = ARRAY_JOIN(FILTER(SPLIT(REGEXP_REPLACE(LOWER(dpk.fn), '[.,]', ' '), ' +'), t -> LENGTH(t) = 1), '')
      )
      -- degenerate short-surname + initial-only given ('L. Li' on a 2000-author
      -- roster): identical ultra-common forms are not identity evidence — abstain
      AND NOT (LENGTH(b.r_last) <= 3 AND LENGTH(b.r_first) = 1 AND LENGTH(dpk.p_first) = 1)
),
unique_donors AS (
    SELECT work_id, author_sequence, raw_name, live_author_id, MAX(donor_id) AS donor_id
    FROM donor_cand
    GROUP BY work_id, author_sequence, raw_name, live_author_id
    HAVING COUNT(DISTINCT donor_id) = 1
),
exclusive AS (
    SELECT *,
        COUNT(*) OVER (PARTITION BY work_id, donor_id) AS donor_claims
    FROM unique_donors
)
SELECT b.work_id, b.author_sequence, b.raw_name, b.author_id AS live_author_id,
       CAST(NULL AS STRING) AS raw_orcid,
       e.donor_id AS rebind_author_id,
       CASE WHEN e.donor_id IS NOT NULL THEN 'onwork_realign' ELSE 'hyper_null' END AS tier,
       b.author_id AS batch_profile_id,
       CAST(NULL AS BIGINT) AS seat_orcid_support,
       CAST(NULL AS BIGINT) AS owner_support
FROM bad b
LEFT JOIN exclusive e
  ON b.work_id = e.work_id AND b.author_sequence = e.author_sequence AND e.donor_claims = 1
-- null tier = FOREIGN-FAMILY only: same/contained/1-edit surnames are name variants
-- (diacritic-transliteration 'Mueller'~'Müller', dropped second surname
-- 'Alcaraz'~'Alcaraz Maestre', dropped initials) — defer those to the judge pass
WHERE e.donor_id IS NOT NULL
   OR (b.r_last != b.p_last
       AND INSTR(b.p_last, b.r_last) = 0 AND INSTR(b.r_last, b.p_last) = 0
       AND levenshtein(b.r_last, b.p_last) > 1)

In [ ]:
-- Batch admission: wave select, curation exclusion, log dedup, profile throttle.
CREATE OR REPLACE TABLE openalex.authors.oxjob608_authority_batch AS
WITH source AS (
    SELECT * FROM openalex.authors.oxjob608_waveO_candidates WHERE repair_wave = 'waveO_orcid_split'
    UNION ALL
    SELECT * FROM openalex.authors.oxjob608_waveD_candidates WHERE repair_wave = 'waveD_displacement'
    UNION ALL
    SELECT * FROM openalex.authors.oxjob608_waveI_candidates WHERE repair_wave = 'waveI_hyper'
),
uncurated AS (
    SELECT s.* FROM source s
    WHERE NOT EXISTS (SELECT 1 FROM openalex.works.work_author_claim_curations cc
                      WHERE cc.work_id = s.work_id)
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_repair_log g
                      WHERE g.work_id = s.work_id AND g.author_sequence = s.author_sequence)
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_authority_repair_log g2
                      WHERE g2.work_id = s.work_id AND g2.author_sequence = s.author_sequence)
),
throttled AS (
    SELECT *, DENSE_RANK() OVER (ORDER BY batch_profile_id) AS prk
    FROM uncurated
)
SELECT work_id, author_sequence, raw_name, live_author_id, raw_orcid,
       rebind_author_id, tier, batch_profile_id, seat_orcid_support, owner_support
FROM throttled WHERE prk <= batch_max_profiles

In [ ]:
-- Gate sample: 150/tier, deterministic hash order.
CREATE OR REPLACE TABLE openalex.authors.oxjob608_authority_gate_sample AS
SELECT b.*,
       COALESCE(oa.display_name, ar.display_name) AS old_profile_name,
       COALESCE(oa2.display_name, ar2.display_name) AS new_profile_name
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY tier ORDER BY xxhash64(work_id, author_sequence)) AS srk
    FROM openalex.authors.oxjob608_authority_batch
) b
LEFT JOIN openalex.authors.openalex_authors oa ON b.live_author_id = oa.id
LEFT JOIN openalex.authors.authors ar ON b.live_author_id = ar.id
LEFT JOIN openalex.authors.openalex_authors oa2 ON b.rebind_author_id = oa2.id
LEFT JOIN openalex.authors.authors ar2 ON b.rebind_author_id = ar2.id
WHERE b.srk <= 150

In [ ]:
-- Opus anchor gate (decider; >=98% per applied tier required before an apply run).
CREATE OR REPLACE TABLE openalex.authors.oxjob608_authority_gate_judged AS
SELECT s.*,
    ai_query(
        'databricks-claude-opus-4-8',
        'You are auditing a repair of author-name to author-profile bindings on scholarly works. '
        || 'A position on a work has an author name string. The OLD profile bound there was judged wrong '
        || CASE WHEN s.raw_orcid IS NOT NULL
                THEN '(this position''s deposited ORCID is NOT the profile''s dominant ORCID: the position''s '
                     || 'ORCID appears on ' || CAST(s.seat_orcid_support AS STRING) || ' of this profile''s positions, '
                     || 'while the NEW profile carries the same ORCID on ' || CAST(s.owner_support AS STRING)
                     || ' of its own positions — authority evidence that this position belongs to the NEW profile, '
                     || 'even when the two profiles'' display names are similar or identical, because ORCID-sharing '
                     || 'name-alike profiles are usually distinct people or deliberate identity splits). '
                WHEN s.tier IN ('onwork_realign', 'hyper_null')
                THEN '(this is a work with 100+ authors whose author list shifted positionally during '
                     || 'ingest; the name at this position is incompatible with the OLD profile''s name'
                     || CASE WHEN s.tier = 'onwork_realign'
                             THEN ', and the NEW profile — whose name matches this position''s name — is itself '
                                  || 'bound at another position of the same work where the name does not match it'
                             ELSE '' END || '). '
                ELSE '(the name is incompatible and the profile''s own name appears at the adjacent position). ' END
        || 'The repair proposes a NEW binding: an existing profile, or NULL meaning unbind and leave '
        || 'the position unattributed for re-matching. '
        || 'Author name at the position: "' || s.raw_name || '". '
        || 'OLD profile display name being removed: "' || COALESCE(s.old_profile_name, '(no profile)') || '". '
        || 'NEW binding: ' || CASE WHEN s.rebind_author_id IS NULL THEN 'NULL (unbind)'
                                   ELSE '"' || COALESCE(s.new_profile_name, '(profile without name)') || '"' END || '. '
        || 'A bare initial is compatible with any full given name starting with that letter, and identical '
        || 'name strings always denote the same person. '
        || 'Answer whether this change is correct-or-harmless: correct when the NEW profile name plausibly '
        || 'denotes the same person as the author name at the position, or when the new binding is NULL and '
        || 'the OLD profile was clearly a different person. Harmful when the OLD profile was actually right.',
        responseFormat => '{"type": "json_schema", "json_schema": {"name": "verdict", "schema": {"type": "object", "properties": {"correct_or_harmless": {"type": "boolean"}, "reason": {"type": "string"}}, "required": ["correct_or_harmless", "reason"], "additionalProperties": false}, "strict": true}}'
    ) AS judge_json
FROM openalex.authors.oxjob608_authority_gate_sample s

In [ ]:
-- Gate readout (run before any apply): must be >=0.98 per tier.
SELECT tier, COUNT(*) AS judged,
       SUM(CASE WHEN judge_json:correct_or_harmless = 'true' THEN 1 ELSE 0 END) AS ok,
       ROUND(SUM(CASE WHEN judge_json:correct_or_harmless = 'true' THEN 1 ELSE 0 END) / COUNT(*), 4) AS precision
FROM openalex.authors.oxjob608_authority_gate_judged
GROUP BY tier ORDER BY judged DESC

In [ ]:
-- Apply step 1 (gated): intent log — every seat about to change, for audit/rollback.
INSERT INTO openalex.authors.oxjob608_authority_repair_log
SELECT current_timestamp(), repair_wave, work_id, author_sequence, raw_name,
       live_author_id, rebind_author_id, tier
FROM openalex.authors.oxjob608_authority_batch
WHERE repair_apply

In [ ]:
-- Apply step 2 (gated): optimistic MERGE — only touches seats still bound as snapshotted.
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT work_id, author_sequence, live_author_id, rebind_author_id
    FROM openalex.authors.oxjob608_authority_batch
    WHERE repair_apply
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id <=> source.live_author_id THEN
    UPDATE SET target.author_id = source.rebind_author_id,
               target.updated_at = current_timestamp()

In [ ]:
-- Apply step 3 (gated): queue repaired works for ES/sync propagation.
INSERT INTO openalex.works.curated_work_ids_pending_sync (work_id, added_datetime)
SELECT DISTINCT b.work_id, current_timestamp()
FROM openalex.authors.oxjob608_authority_batch b
WHERE repair_apply
  AND NOT EXISTS (SELECT 1 FROM openalex.works.curated_work_ids_pending_sync pending
                  WHERE pending.work_id = b.work_id)

In [ ]:
-- Verify: diverged count (seats whose live binding no longer matches what we applied).
SELECT COUNT(*) AS diverged
FROM openalex.authors.oxjob608_authority_repair_log g
JOIN openalex.works.work_authors wa
  ON g.work_id = wa.work_id AND g.author_sequence = wa.author_sequence
WHERE g.wave = repair_wave
  AND g.applied_at >= current_timestamp() - INTERVAL 1 HOUR
  AND NOT (wa.author_id <=> g.new_author_id)

In [ ]:
-- Verify: batch summary for the operator log.
SELECT repair_wave AS wave, tier, COUNT(*) AS seats,
       COUNT(DISTINCT batch_profile_id) AS profiles,
       COUNT(DISTINCT work_id) AS works,
       SUM(CASE WHEN rebind_author_id IS NULL THEN 1 ELSE 0 END) AS nulls
FROM openalex.authors.oxjob608_authority_batch
GROUP BY tier ORDER BY seats DESC